# 🧩 Mini-Lab: Pin Dependencies

**Module: Escape the Notebook: Environment, Repo, Git** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Explain** why pinning exact dependency versions is critical for reproducibility
2. **Create** a `requirements.txt` with pinned versions using `pip freeze`
3. **Regenerate** a clean environment from a pinned requirements file

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Dependency Pinning | Lock your project’s libraries to specific versions so installs are repeatable, bugs are reproducible, and deployments behave the same everywhere. |
| Virtual Environments | A virtual environment is an isolated workspace that keeps a project’s Python packages and settings separate, so installs and upgrades do not break other projects. |


## Why Pinning Matters

Recall from **mini-create-virtualenv**: a virtual environment gives your project its own isolated `site-packages`. But isolation alone isn't enough.

When you run `pip install requests`, pip grabs the **latest** version. A month later, a colleague clones your repo and runs the same command — they get a **different** version. If the library changed its API or introduced a bug, your project breaks on their machine.

**Pinning** means writing down the exact version of every package so that `pip install` always produces the same environment.

In [1]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


# Show the contrast: unpinned vs pinned
comparison = """
### Unpinned vs Pinned

| Style | `requirements.txt` | What `pip install -r` does | Risk |
|-------|-------------------|---------------------------|------|
| **Unpinned** | `requests` | Installs latest (e.g., 2.32.0 today, 2.33.0 next month) | Different versions on different machines |
| **Loose range** | `requests>=2.28` | Installs 2.28 or higher — could be anything | Still non-deterministic |
| **Pinned** | `requests==2.31.0` | Always installs exactly 2.31.0 | ✅ Same environment every time |
"""

md(comparison)


### Unpinned vs Pinned

| Style | `requirements.txt` | What `pip install -r` does | Risk |
|-------|-------------------|---------------------------|------|
| **Unpinned** | `requests` | Installs latest (e.g., 2.32.0 today, 2.33.0 next month) | Different versions on different machines |
| **Loose range** | `requests>=2.28` | Installs 2.28 or higher — could be anything | Still non-deterministic |
| **Pinned** | `requests==2.31.0` | Always installs exactly 2.31.0 | ✅ Same environment every time |


## The Pinning Workflow

The standard workflow has two sides: **you** (the developer) pin what you installed, and **the newcomer** (the stranger cloning your repo) installs from your pin file.

All of the commands below run **in the terminal** with your virtual environment activated (recall `source .venv/bin/activate` or `.venv\Scripts\activate`).

### Step 1 — Install packages you need

With your virtual environment activated, install the packages your project depends on. For example:

```bash
pip install requests openai python-dotenv
```

At this point, pip installs the latest compatible versions and their transitive dependencies (packages that *your* packages depend on).

### Step 2 — Freeze the exact versions

`pip freeze` outputs every installed package with its exact version. Redirect it to a file:

```bash
pip freeze > requirements.txt
```

This creates (or overwrites) `requirements.txt` with lines like:

```
certifi==2024.2.2
charset-normalizer==3.3.2
idna==3.6
openai==1.30.1
python-dotenv==1.0.1
requests==2.31.0
urllib3==2.2.1
```

Notice: **every** package is pinned with `==`, including transitive dependencies you didn't install directly. This is the "full snapshot" approach — it guarantees identical installs.

### Step 3 — Commit `requirements.txt` to your repo

This file goes into version control. It's the answer to the newcomer's question: *"What packages do I need, and which versions?"*

## The Newcomer's Side: Regenerating the Environment

When someone clones your repo, they create a fresh virtual environment and install from your pin file. Here's the complete sequence:

```bash
git clone <repo-url>
cd your-project
python -m venv .venv
source .venv/bin/activate      # macOS/Linux
# .venv\Scripts\activate       # Windows
pip install -r requirements.txt
```

Because every line in `requirements.txt` uses `==`, pip installs **exactly** the same versions you had. The environment is now a clone of yours.

In [2]:
# Let's simulate what pip freeze output looks like
# and verify it uses the == format

simulated_freeze = """certifi==2024.2.2
charset-normalizer==3.3.2
idna==3.6
openai==1.30.1
python-dotenv==1.0.1
requests==2.31.0
urllib3==2.2.1"""

lines = simulated_freeze.strip().split("\n")
all_pinned = all("==" in line for line in lines)

print(f"Total packages: {len(lines)}")
print(f"All pinned with '==': {all_pinned}")
print()
for line in lines:
    pkg, version = line.split("==")
    print(f"  {pkg:<25} → {version}")

Total packages: 7
All pinned with '==': True

  certifi                   → 2024.2.2
  charset-normalizer        → 3.3.2
  idna                      → 3.6
  openai                    → 1.30.1
  python-dotenv             → 1.0.1
  requests                  → 2.31.0
  urllib3                   → 2.2.1


## What Can Go Wrong Without Pinning

Let's look at a concrete scenario to make the risk tangible.

In [3]:
# A timeline showing how unpinned dependencies break projects

timeline = """
### The Version Drift Problem

| When | What Happens | `some-lib` Version | Result |
|------|-------------|-------------------|--------|
| **January** | You build & test the project | 1.4.2 | ✅ Works |
| **March** | Library releases 1.5.0 (renames a function) | 1.5.0 | — |
| **April** | Colleague clones, runs `pip install some-lib` | 1.5.0 | ❌ `ImportError` |

With `requirements.txt` containing `some-lib==1.4.2`, the colleague would have installed **1.4.2** — the version you tested — and the project would work.

> **Version drift** is the #1 reason "works on my machine" happens. Pinning eliminates it.
"""

md(timeline)


### The Version Drift Problem

| When | What Happens | `some-lib` Version | Result |
|------|-------------|-------------------|--------|
| **January** | You build & test the project | 1.4.2 | ✅ Works |
| **March** | Library releases 1.5.0 (renames a function) | 1.5.0 | — |
| **April** | Colleague clones, runs `pip install some-lib` | 1.5.0 | ❌ `ImportError` |

With `requirements.txt` containing `some-lib==1.4.2`, the colleague would have installed **1.4.2** — the version you tested — and the project would work.

> **Version drift** is the #1 reason "works on my machine" happens. Pinning eliminates it.


## Updating Pinned Dependencies

Pinning doesn't mean you never update. When you intentionally want to upgrade a package:

```bash
pip install --upgrade requests
pip freeze > requirements.txt
```

Then test that your project still works with the new version. If it does, commit the updated `requirements.txt`. If it doesn't, roll back.

The point is that version changes are **deliberate and tracked**, not accidental.

## Quick-Reference Cheat Sheet

In [4]:
cheat_sheet = """
### Dependency Pinning Commands

| Action | Command |
|--------|---------|
| **Install a package** | `pip install <package>` |
| **Freeze all versions** | `pip freeze > requirements.txt` |
| **Install from pin file** | `pip install -r requirements.txt` |
| **Upgrade a package** | `pip install --upgrade <package>` |
| **Re-freeze after upgrade** | `pip freeze > requirements.txt` |
| **See what's installed** | `pip list` |

### The Full Reproducibility Sequence (for the newcomer)

```bash
git clone <repo-url>
cd project
python -m venv .venv
source .venv/bin/activate   # or .venv\\Scripts\\activate on Windows
pip install -r requirements.txt
python src/app.py           # run the project
```

> **Tip:** Always freeze *after* you've tested that things work. The pin file is a snapshot of a known-good state.
"""

md(cheat_sheet)


### Dependency Pinning Commands

| Action | Command |
|--------|---------|
| **Install a package** | `pip install <package>` |
| **Freeze all versions** | `pip freeze > requirements.txt` |
| **Install from pin file** | `pip install -r requirements.txt` |
| **Upgrade a package** | `pip install --upgrade <package>` |
| **Re-freeze after upgrade** | `pip freeze > requirements.txt` |
| **See what's installed** | `pip list` |

### The Full Reproducibility Sequence (for the newcomer)

```bash
git clone <repo-url>
cd project
python -m venv .venv
source .venv/bin/activate   # or .venv\Scripts\activate on Windows
pip install -r requirements.txt
python src/app.py           # run the project
```

> **Tip:** Always freeze *after* you've tested that things work. The pin file is a snapshot of a known-good state.


## 🎯 Summary

### Key Takeaways

1. **`pip freeze > requirements.txt`** — captures every installed package with its exact version, creating a reproducible snapshot of your environment
2. **`pip install -r requirements.txt`** — recreates that exact environment in a fresh virtual environment, eliminating version drift
3. **Pinning prevents "works on my machine"** — without exact versions, `pip install` grabs the latest release, which may be different from what you tested
4. **Updates should be deliberate** — upgrade a package, test it, then re-freeze; version changes are tracked in version control, not accidental